# Chapter 21: Model Selection and Hyperparameter Optimization

## Learning Objectives\n- Select models using fair validation protocols\n- Run random search efficiently\n- Keep tuning separate from final evaluation\n- Track experiments reproducibly

## Prerequisites\n- Python basics and functions\n- Numpy basics for deep learning chapters\n- Understanding of earlier chapters (0-19)\n

# Chapter 21: Model Selection and Hyperparameter Optimization

## Why this chapter matters
Advanced ML projects fail when model selection is done with weak validation design.
This chapter gives a repeatable process used in production teams.

## Learning goals
1. Use nested cross-validation correctly.
2. Separate model selection from final performance estimation.
3. Run random search and Bayesian-style intuition (without heavy dependencies).
4. Track experiments and compare fairly.

## Step 1: Define the objective and constraints
A model is selected by:
- primary metric (e.g., F1, AUC, RMSE)
- latency budget
- memory budget
- explainability requirements

## Step 2: Validation protocols
1. Standard k-fold for IID data.
2. Group k-fold for entity leakage risk.
3. Time-based split for temporal data.

## Step 3: Nested CV
Outer loop: unbiased estimate.
Inner loop: choose hyperparameters.

Never tune and report on the same folds.

## Step 4: Search strategy
Preferred order:
1. random search on broad ranges
2. focused random search on narrowed ranges
3. optional Bayesian optimizer later

## Step 5: Experiment tracking minimum fields
- run id
- code version
- feature version
- hyperparameters
- metrics by split
- training time
- artifact path

## Assignment
1. Compare Logistic Regression, CART, and Naive Bayes.
2. Use the same split protocol.
3. Run random search (30+ configs per model).
4. Pick champion model with metric + latency criteria.
5. Write a model card.



## Checkpoint\n\n1. You can explain nested CV and why it matters.\n2. You can define objective + latency constraints.\n3. You can summarize random search tradeoffs.

In [ ]:
"""Simple random search runner (framework-agnostic).

Replace `train_and_eval` with your model training function.
"""

from __future__ import annotations

import random
import time
from typing import Dict, List, Tuple


def sample_space() -> Dict[str, float]:
    return {
        "learning_rate": 10 ** random.uniform(-4, -1),
        "l2": 10 ** random.uniform(-6, -2),
        "epochs": random.choice([50, 100, 150, 200]),
    }


def train_and_eval(params: Dict[str, float]) -> Tuple[float, float]:
    # TODO: plug in your actual training pipeline and return (score, latency_ms)
    # This placeholder mimics a score surface with noise.
    score = 0.8 - abs(params["learning_rate"] - 0.01) * 2.0
    score -= abs(params["l2"] - 1e-4) * 20.0
    score += random.uniform(-0.01, 0.01)
    latency_ms = random.uniform(2.0, 8.0)
    return score, latency_ms


def random_search(n_trials: int = 40) -> List[Dict[str, float]]:
    history = []
    for trial in range(1, n_trials + 1):
        params = sample_space()
        t0 = time.time()
        score, latency_ms = train_and_eval(params)
        elapsed = (time.time() - t0) * 1000.0

        row = {
            "trial": trial,
            "score": score,
            "latency_ms": latency_ms,
            "eval_ms": elapsed,
            **params,
        }
        history.append(row)

    history.sort(key=lambda x: x["score"], reverse=True)
    return history


def main() -> None:
    results = random_search(40)
    print("Top 5 runs:")
    for r in results[:5]:
        print(r)


if __name__ == "__main__":
    main()


## Guided Exercise\nModify the search space to include model depth and compare top runs by both score and latency.

In [ ]:
# TODO (Student Exercise)
results = random_search(20)
print(results[:3])

## Exercise Solution

In [ ]:
results = random_search(40)
feasible = [r for r in results if r['latency_ms'] <= 5.0]
print('feasible top3:')
for row in feasible[:3]:
    print(row)

## Quick Quiz\n\n**Q1.** Why not tune and report on the same folds?\n\n**Answer:** It gives optimistically biased performance.\n\n**Q2.** When is group-based splitting required?\n\n**Answer:** When rows share entities and leakage risk exists.\n\n**Q3.** Why random search before grid search in large spaces?\n\n**Answer:** It covers broad ranges more efficiently.\n

## Homework\nRun at least 30 configurations for 2 different model families and write a champion-selection memo.